# Phishing Website Detection

**Goal:** Build and compare machine-learning classifiers that distinguish phishing and legitimate websites using website/URL-related features.

**Dataset source:** Kaggle — *Phishing Website Detector*  
Expected file: `data/phishing.csv`

The dataset is commonly reported as having 11,054 records and roughly 30 website-related predictor features plus identifiers/target fields.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

DATA_PATH = Path("data/phishing.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Dataset not found. Download phishing.csv from the Kaggle source listed in README.md "
        "and place it inside the data/ folder."
    )

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()


## 1. Data Understanding


In [ ]:
print("Columns:")
for i, c in enumerate(df.columns, 1):
    print(f"{i:>2}. {c}")

print("\nDuplicate rows:", df.duplicated().sum())
print("\nMissing values:", int(df.isna().sum().sum()))
display(df.describe(include="all").T)


## 2. Target Identification and Cleaning


In [ ]:
data = df.copy()

# Remove common identifier columns if present
for c in ["Index", "index", "id", "ID"]:
    if c in data.columns:
        data = data.drop(columns=c)

target_candidates = ["class", "Class", "status", "Status", "label", "Label", "Result", "result"]
target_col = next((c for c in target_candidates if c in data.columns), None)

if target_col is None:
    raise ValueError(
        "Target column not detected. Check the dataset columns and set target_col manually."
    )

print("Target column:", target_col)
print(data[target_col].value_counts(dropna=False))


In [ ]:
# Convert target to 0/1 where needed.
# For the commonly used Kaggle file, labels may use -1 and 1.
y_raw = data[target_col].copy()

unique_vals = set(pd.Series(y_raw.dropna().unique()).tolist())
print("Original target values:", unique_vals)

if unique_vals.issubset({-1, 1}):
    # Keep 1 as positive class and map -1 to 0.
    y = y_raw.map({-1: 0, 1: 1})
elif unique_vals.issubset({0, 1}):
    y = y_raw.astype(int)
else:
    # Factorize only as a fallback; inspect mapping before interpreting results.
    codes, uniques = pd.factorize(y_raw)
    y = pd.Series(codes, index=y_raw.index)
    print("Fallback target mapping:", dict(enumerate(uniques)))

X = data.drop(columns=[target_col]).copy()

# Keep only numeric columns for this classical ML workflow
X = X.apply(pd.to_numeric, errors="coerce")
X = X.fillna(X.median(numeric_only=True))

print("Feature matrix:", X.shape)
print("Target distribution:")
display(y.value_counts().rename("count").to_frame())


## 3. Class Distribution


In [ ]:
counts = y.value_counts().sort_index()
plt.figure(figsize=(6,4))
plt.bar(counts.index.astype(str), counts.values)
plt.title("Target Class Distribution")
plt.xlabel("Class")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


## 4. Train/Test Split


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)


## 5. Model Comparison


In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, random_state=42))
    ]),
    "KNN": Pipeline([
        ("scale", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=5))
    ]),
    "Decision Tree": DecisionTreeClassifier(random_state=42, max_depth=12),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, random_state=42, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

results = []
fitted = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    fitted[name] = model

    pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, proba)
    else:
        auc = np.nan

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "ROC_AUC": auc
    })

results_df = pd.DataFrame(results).sort_values("F1", ascending=False)
results_df


In [ ]:
plot_df = results_df.set_index("Model")[["Accuracy", "Precision", "Recall", "F1"]]
ax = plot_df.plot(kind="bar", figsize=(10,5))
ax.set_ylim(0, 1.05)
ax.set_title("Model Performance Comparison")
ax.set_ylabel("Score")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()


## 6. Best Model Evaluation


In [ ]:
best_name = results_df.iloc[0]["Model"]
best_model = fitted[best_name]
best_pred = best_model.predict(X_test)

print("Best model by F1:", best_name)
print("\nClassification report:\n")
print(classification_report(y_test, best_pred, zero_division=0))

cm = confusion_matrix(y_test, best_pred)
print("Confusion matrix:\n", cm)


In [ ]:
cm = confusion_matrix(y_test, best_pred)

plt.figure(figsize=(5,4))
plt.imshow(cm)
plt.title(f"Confusion Matrix — {best_name}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks([0,1], ["0","1"])
plt.yticks([0,1], ["0","1"])

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.tight_layout()
plt.show()


## 7. Feature Importance


In [ ]:
rf = fitted["Random Forest"]
importance = (
    pd.Series(rf.feature_importances_, index=X.columns)
      .sort_values(ascending=False)
      .head(15)
)

display(importance.to_frame("importance"))

plt.figure(figsize=(8,6))
plt.barh(importance.sort_values().index, importance.sort_values().values)
plt.title("Top 15 Random Forest Feature Importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()


## 8. Portfolio Conclusions

After running the notebook, write the final findings using the actual output:

- Which model achieved the strongest F1 / accuracy?
- Which features were most important?
- Did the classes appear balanced?
- What kinds of errors matter most for phishing detection?
- How could the model be improved or deployed safely?

> Avoid claiming a model is production-ready based only on notebook accuracy. Real-world phishing detection needs fresh data, monitoring, and security validation.
